# SQL Analysis — Cart Abandonment
**Project:** Reducing Cart Abandonment at an Undisclosed E-Commerce Platform
**Dataset:** E-Commerce Behavior Data — October 2019 (Open CDP via Kaggle)

## 1. Setup — Connect to SQLite Database

In [1]:
import sqlite3
import pandas as pd

# Connect to existing database
conn = sqlite3.connect('data/ecommerce.db')

# Verify connection
print(f"Connected")
print(f"Rows in events table: {conn.execute('SELECT COUNT(*) FROM events').fetchone()[0]:,}")
print(f"Columns: {[row[1] for row in conn.execute('PRAGMA table_info(events)').fetchall()]}")

Connected
Rows in events table: 5,084,161
Columns: ['event_time', 'event_type', 'product_id', 'category_id', 'category_code', 'brand', 'price', 'user_id', 'user_session', 'hour', 'day_of_week', 'date']


## 2. SQL Queries

### 2.1 Funnel Analysis — Sessions at Each Stage

In [2]:
query = """
SELECT
    event_type,
    COUNT(DISTINCT user_session) AS unique_sessions,
    ROUND(COUNT(DISTINCT user_session) * 100.0 / 
        (SELECT COUNT(DISTINCT user_session) FROM events WHERE event_type = 'view'), 2) AS pct_of_views
FROM events
GROUP BY event_type
ORDER BY unique_sessions DESC
"""
pd.read_sql_query(query, conn)

,event_type,unique_sessions,pct_of_views
0,view,1109110,100.00
1,purchase,75541,6.81
2,cart,68707,6.19


### 2.2 Cart Abandonment Rate

In [3]:
query = """
WITH cart AS (
    SELECT COUNT(DISTINCT user_session) AS total FROM events WHERE event_type = 'cart'
),
purchased AS (
    SELECT COUNT(DISTINCT user_session) AS total FROM events 
    WHERE event_type = 'purchase'
    AND user_session IN (SELECT DISTINCT user_session FROM events WHERE event_type = 'cart')
)
SELECT
    cart.total AS cart_sessions,
    purchased.total AS converted_sessions,
    cart.total - purchased.total AS abandoned_sessions,
    ROUND((cart.total - purchased.total) * 100.0 / cart.total, 2) AS abandonment_rate_pct,
    ROUND(purchased.total * 100.0 / cart.total, 2) AS conversion_rate_pct
FROM cart, purchased
"""
pd.read_sql_query(query, conn)

,cart_sessions,converted_sessions,abandoned_sessions,abandonment_rate_pct,conversion_rate_pct
0,68707,34899,33808,49.21,50.79


### 2.3 Abandonment Rate by Category

In [4]:
query = """
WITH purchase_sess AS (
    SELECT DISTINCT user_session FROM events WHERE event_type = 'purchase'
),
category_stats AS (
    SELECT
        SUBSTR(category_code, 1, INSTR(category_code || '.', '.') - 1) AS category,
        COUNT(DISTINCT user_session) AS total_carts,
        COUNT(DISTINCT CASE WHEN user_session IN (SELECT user_session FROM purchase_sess) 
              THEN user_session END) AS converted
    FROM events
    WHERE event_type = 'cart'
    AND category_code IS NOT NULL
    GROUP BY category
    HAVING COUNT(DISTINCT user_session) > 100
)
SELECT
    category,
    total_carts,
    converted,
    total_carts - converted AS abandoned,
    ROUND((total_carts - converted) * 100.0 / total_carts, 2) AS abandonment_rate_pct
FROM category_stats
ORDER BY abandonment_rate_pct DESC
"""
pd.read_sql_query(query, conn)

,category,total_carts,converted,abandoned,abandonment_rate_pct
0,kids,190,58,132,69.47
1,auto,735,317,418,56.87
2,construction,797,356,441,55.33
3,appliances,6969,3155,3814,54.73
4,computers,2169,992,1177,54.26
5,furniture,206,99,107,51.94
6,electronics,49714,26883,22831,45.92


### 2.4 Abandonment Rate by Hour of Day

In [5]:
query = """
WITH purchase_sess AS (
    SELECT DISTINCT user_session FROM events WHERE event_type = 'purchase'
)
SELECT
    hour,
    COUNT(DISTINCT user_session) AS total_carts,
    COUNT(DISTINCT CASE WHEN user_session IN (SELECT user_session FROM purchase_sess) 
          THEN user_session END) AS converted,
    COUNT(DISTINCT user_session) - COUNT(DISTINCT CASE WHEN user_session IN 
          (SELECT user_session FROM purchase_sess) THEN user_session END) AS abandoned,
    ROUND((COUNT(DISTINCT user_session) - COUNT(DISTINCT CASE WHEN user_session IN 
          (SELECT user_session FROM purchase_sess) THEN user_session END)) * 100.0 / 
          COUNT(DISTINCT user_session), 2) AS abandonment_rate_pct
FROM events
WHERE event_type = 'cart'
GROUP BY hour
ORDER BY hour
"""
pd.read_sql_query(query, conn)

,hour,total_carts,converted,abandoned,abandonment_rate_pct
0,0,328,133,195,59.45
1,1,602,238,364,60.47
2,2,1404,623,781,55.63
3,3,2733,1427,1306,47.79
4,4,3658,1955,1703,46.56
5,5,4383,2308,2075,47.34
6,6,4831,2596,2235,46.26
7,7,4922,2656,2266,46.04
8,8,4958,2689,2269,45.76
9,9,5081,2764,2317,45.60


### 2.5 Abandonment Rate by Day of Week

In [6]:
query = """
WITH purchase_sess AS (
    SELECT DISTINCT user_session FROM events WHERE event_type = 'purchase'
)
SELECT
    day_of_week,
    COUNT(DISTINCT user_session) AS total_carts,
    COUNT(DISTINCT CASE WHEN user_session IN (SELECT user_session FROM purchase_sess) 
          THEN user_session END) AS converted,
    ROUND((COUNT(DISTINCT user_session) - COUNT(DISTINCT CASE WHEN user_session IN 
          (SELECT user_session FROM purchase_sess) THEN user_session END)) * 100.0 / 
          COUNT(DISTINCT user_session), 2) AS abandonment_rate_pct
FROM events
WHERE event_type = 'cart'
GROUP BY day_of_week
ORDER BY abandonment_rate_pct DESC
"""
pd.read_sql_query(query, conn)

,day_of_week,total_carts,converted,abandonment_rate_pct
0,Friday,10449,5087,51.32
1,Saturday,9794,4807,50.92
2,Tuesday,10231,5102,50.13
3,Sunday,9608,4937,48.62
4,Monday,8723,4520,48.18
5,Wednesday,10082,5286,47.57
6,Thursday,9838,5172,47.43


### 2.6 Top 10 Most Abandoned Products by Revenue Lost

In [7]:
query = """
WITH purchase_sess AS (
    SELECT DISTINCT user_session FROM events WHERE event_type = 'purchase'
)
SELECT
    product_id,
    category_code,
    brand,
    price,
    COUNT(*) AS times_abandoned,
    ROUND(COUNT(*) * price, 2) AS potential_revenue_lost
FROM events
WHERE event_type = 'cart'
AND user_session NOT IN (SELECT user_session FROM purchase_sess)
AND brand IS NOT NULL
GROUP BY product_id, category_code, brand, price
ORDER BY potential_revenue_lost DESC
LIMIT 10
"""
pd.read_sql_query(query, conn)

,product_id,category_code,brand,price,times_abandoned,potential_revenue_lost
0,1005115,electronics.smartphone,apple,975.56,225,219501.00
1,1005105,electronics.smartphone,apple,1415.48,109,154287.32
2,1005135,electronics.smartphone,apple,1747.79,62,108362.98
3,1005105,electronics.smartphone,apple,1428.31,73,104266.63
4,1005115,electronics.smartphone,apple,1003.85,97,97373.45
5,1005135,electronics.smartphone,apple,1741.34,51,88808.34
6,1004249,electronics.smartphone,apple,766.76,113,86643.88
7,1801690,electronics.video.tv,samsung,369.37,222,82000.14
8,1002633,electronics.smartphone,apple,358.57,224,80319.68
9,1005135,electronics.smartphone,apple,1725.03,40,69001.20


### 2.7 Running Total of Revenue Lost Per Day

In [8]:
query = """
WITH purchase_sess AS (
    SELECT DISTINCT user_session FROM events WHERE event_type = 'purchase'
),
daily_lost AS (
    SELECT
        date AS event_date,
        ROUND(SUM(price), 2) AS daily_revenue_lost
    FROM events
    WHERE event_type = 'cart'
    AND user_session NOT IN (SELECT user_session FROM purchase_sess)
    GROUP BY date
)
SELECT
    event_date,
    daily_revenue_lost,
    ROUND(SUM(daily_revenue_lost) OVER (ORDER BY event_date), 2) AS running_total_lost
FROM daily_lost
ORDER BY event_date
"""
pd.read_sql_query(query, conn)

,event_date,daily_revenue_lost,running_total_lost
0,2019-10-01,267067.82,267067.82
1,2019-10-02,287595.27,554663.09
2,2019-10-03,379038.69,933701.78
3,2019-10-04,837894.26,1771596.04
4,2019-10-05,726859.01,2498455.05
5,2019-10-06,586118.76,3084573.81
6,2019-10-07,278646.35,3363220.16
7,2019-10-08,267513.50,3630733.66
8,2019-10-09,262877.57,3893611.23
9,2019-10-10,329538.43,4223149.66


### 2.8 Users Who Added Same Product to Cart Multiple Times Without Buying

In [9]:
query = """
WITH purchase_sess AS (
    SELECT DISTINCT user_session FROM events WHERE event_type = 'purchase'
),
repeat_abandoners AS (
    SELECT
        user_id,
        product_id,
        brand,
        price,
        COUNT(*) AS times_carted
    FROM events
    WHERE event_type = 'cart'
    AND user_session NOT IN (SELECT user_session FROM purchase_sess)
    AND brand IS NOT NULL
    GROUP BY user_id, product_id, brand, price
    HAVING COUNT(*) > 1
)
SELECT
    product_id,
    brand,
    price,
    COUNT(DISTINCT user_id) AS users_repeatedly_abandoning,
    ROUND(AVG(times_carted), 1) AS avg_times_carted_per_user
FROM repeat_abandoners
GROUP BY product_id, brand, price
ORDER BY users_repeatedly_abandoning DESC
LIMIT 10
"""
pd.read_sql_query(query, conn)

,product_id,brand,price,users_repeatedly_abandoning,avg_times_carted_per_user
0,1004856,samsung,131.64,62,3.0
1,1004856,samsung,131.53,57,2.6
2,1004856,samsung,130.99,48,3.4
3,1004741,xiaomi,189.97,40,2.7
4,1004777,xiaomi,135.01,37,2.6
5,1002633,apple,358.57,34,2.5
6,1004856,samsung,131.51,33,2.7
7,1004767,samsung,246.52,31,3.2
8,1801690,samsung,369.37,30,3.3
9,1005115,apple,975.56,29,2.8


### 2.9 Average Session Length — Converters vs Abandoners

In [10]:
query = """
WITH purchase_sess AS (
    SELECT DISTINCT user_session FROM events WHERE event_type = 'purchase'
)
SELECT
    CASE WHEN user_session IN (SELECT user_session FROM purchase_sess) 
         THEN 'Converted' ELSE 'Abandoned' END AS session_type,
    COUNT(DISTINCT user_session) AS total_sessions,
    ROUND(AVG(event_count), 1) AS avg_events_per_session,
    MIN(event_count) AS min_events,
    MAX(event_count) AS max_events
FROM (
    SELECT user_session, COUNT(*) AS event_count
    FROM events
    GROUP BY user_session
)
GROUP BY session_type
"""
pd.read_sql_query(query, conn)

,session_type,total_sessions,avg_events_per_session,min_events,max_events
0,Abandoned,1033790,4.4,1,293
1,Converted,75541,7.5,1,196


### 2.10 Top 10 Brands by Total Cart Value

In [11]:
query = """
SELECT
    brand,
    COUNT(*) AS total_cart_events,
    ROUND(SUM(price), 2) AS total_cart_value,
    ROUND(AVG(price), 2) AS avg_cart_price
FROM events
WHERE event_type = 'cart'
AND brand IS NOT NULL
GROUP BY brand
HAVING COUNT(*) > 1000
ORDER BY total_cart_value DESC
LIMIT 10
"""
pd.read_sql_query(query, conn)

,brand,total_cart_events,total_cart_value,avg_cart_price
0,apple,24470,18684003.20,763.55
1,samsung,34895,8971493.02,257.10
2,xiaomi,12059,1769669.60,146.75
3,huawei,4952,1035178.67,209.04
4,acer,1083,591235.31,545.92
5,lg,1500,577184.48,384.79
6,oppo,2339,529364.34,226.32
7,sony,1168,415021.22,355.33


### 2.11 Abandonment Rate by Brand

In [13]:
query = """
WITH purchase_sess AS (
    SELECT DISTINCT user_session FROM events WHERE event_type = 'purchase'
)
SELECT
    brand,
    COUNT(DISTINCT user_session) AS total_carts,
    COUNT(DISTINCT CASE WHEN user_session IN (SELECT user_session FROM purchase_sess) 
          THEN user_session END) AS converted,
    COUNT(DISTINCT user_session) - COUNT(DISTINCT CASE WHEN user_session IN 
          (SELECT user_session FROM purchase_sess) THEN user_session END) AS abandoned,
    ROUND((COUNT(DISTINCT user_session) - COUNT(DISTINCT CASE WHEN user_session IN 
          (SELECT user_session FROM purchase_sess) THEN user_session END)) * 100.0 / 
          COUNT(DISTINCT user_session), 2) AS abandonment_rate_pct
FROM events
WHERE event_type = 'cart'
AND brand IS NOT NULL
GROUP BY brand
HAVING COUNT(DISTINCT user_session) > 500
ORDER BY abandonment_rate_pct DESC
LIMIT 15
"""
pd.read_sql_query(query, conn)

,brand,total_carts,converted,abandoned,abandonment_rate_pct
0,xiaomi,7591,3252,4339,57.16
1,lg,1030,446,584,56.70
2,bosch,564,245,319,56.56
3,sony,817,359,458,56.06
4,elenberg,704,330,374,53.13
5,acer,751,376,375,49.93
6,philips,540,272,268,49.63
7,apple,16345,8854,7491,45.83
8,huawei,3222,1771,1451,45.03
9,samsung,22324,12928,9396,42.09
